# Approach: Complete Collaboration + HT-MIA

In this approach, our model is a mash-up of the parts of the previous models. We are using the pipeline and 9 base features from the first model. Then, we use Min-K% features, cross-model delta attack, in additon to the new features using HT-MIA. Finally, we train classifiers using logistic regression, random forest, XGBoost, Gradient Boosting. XGBoost gave the best results. Future models may consider removing logistic regression and random forest.

In [ ]:
import sys
# Path to the model5 directory after uploading to Colab.
# If you uploaded to /content/, this is correct.
sys.path.insert(0, '/content/model5')


#mount drive
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/model5/"




In [ ]:
#imports

import pickle
import warnings
warnings.filterwarnings('ignore')
import torch
from models import load_data, load_models_and_tokenizer
from batches import process_all_splits, save_features, load_features
from build_membership_classifier import build_membership_classifier, compute_zlib_baseline, generate_submission


In [ ]:
#use gpu if available

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Always run this — splits are needed even when loading cached features.
splits = load_data()

In [ ]:
# Skip this cell if loading from cached features.
#we loading the tokenizer and 4 huggingface models fine-tuned model_135m, model_360m, as well as their reference models: ref_model_135m and ref_model_360m
tokenizer, model_135m, model_360m, ref_model_135m, ref_model_360m = load_models_and_tokenizer(device)

In [ ]:
#perform feature extraction over train, validation, test on all models
#reference models have to be passed to the function so that they can be used to establish a baseline for HT-MIA to be calculated  
features = process_all_splits(
    model_135m, model_360m, ref_model_135m, ref_model_360m,
    tokenizer, splits, batch_size=500,
)

In [ ]:
#Features are saved to file
save_features(SAVE_DIR, features)

In [ ]:
# Run this cell instead of the extraction cells when resuming a session with saved features.
#features = load_features(SAVE_DIR)

In [ ]:
#Build classifier using full training set and evaluate on validation set.
results = build_membership_classifier(
    features["train_135m"], features["train_360m"],
    features["val_135m"],   features["val_360m"],
)

In [ ]:
# Results summary
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 50)

zlib_result = compute_zlib_baseline(splits["validation"])

for name, r in results.items():
    print(f"\n{name}:  AUC={r['auc']:.4f}  TPR@FPR=0.1={r['tpr_at_fpr_01']:.4f}  Acc={r['accuracy']:.4f}")

#Get the best classifier
best_name = max(results, key=lambda x: results[x]["auc"])
best      = results[best_name]
print(f"\nBest: {best_name}  AUC={best['auc']:.4f}  TPR@FPR=0.1={best['tpr_at_fpr_01']:.4f}")

# Save classifier to Google Drive
classifier_path = SAVE_DIR + "membership_classifier.pkl"
with open(classifier_path, "wb") as f:
    pickle.dump(best, f)
print(f"Saved classifier to {classifier_path}")

# Generate Kaggle submission — saved locally so files.download can pick it up
submission_path = "membership_inference_submission.csv"
submission_df = generate_submission(
    features["test_135m"], features["test_360m"],
    best, splits["test"],
    output_path=submission_path,
)
print(submission_df.head())

In [ ]:
#Download Kaggle Submission File
from google.colab import files
files.download('membership_inference_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>